## Parte 1- Scraping de decretos de emergencia por lluvias

Reunimos los decretos supremos de la PCM publicados entre el 1 de enero y el 31 de mayo de 2025.

In [138]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

**Paso 1: Robots**

In [139]:
respuesta_robots = requests.get(
    "https://www.gob.pe/robots.txt",
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)
print("Estado:", respuesta_robots.status_code)
print(respuesta_robots.text)

Estado: 200
# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



**Qué prohíbe:**
Para todos los robots (`User-agent: *`), el sitio prohíbe el acceso a `/admin/` y a las URL que contienen el parámetro `sheet=`. Para `AhrefsBot` prohíbe todo el sitio (`Disallow: /`). Las líneas que bloquearían a todos los robots están comentadas con `#`, así que no se aplican.

**¿Está permitida /busquedas?**
Sí. La ruta `/busquedas` no aparece en ninguna regla `Disallow` y nuestra URL no contiene el parámetro `sheet=`, así que el robots.txt no prohíbe rastrearla.

**¿Por qué igual hacemos pausas?**
Para los robots en general (`User-agent: *`) no se fija un `Crawl-delay`, pero a otros bots sí se les pide: 5 segundos a GPTBot y 2 segundos a OAI-SearchBot. Ese es el ritmo que el sitio considera razonable. gob.pe es un servicio público: si le mandamos pedidos seguidos lo cargamos sin necesidad y nos pueden bloquear. Por eso esperamos 2 s entre páginas de búsqueda y 1 s entre páginas de normas.


**Paso 2 · Abrir un mes con Selenium**

In [140]:
opciones = Options()
# opciones.add_argument("--headless=new")   # descomenten para no ver la ventana
opciones.add_argument("--window-size=1400,900")

navegador = webdriver.Chrome(options=opciones)

In [141]:
URL_BASE = (
    "https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent"
)

navegador.get(URL_BASE + "&desde=01-01-2025&hasta=31-01-2025")

# Los resultados los dibuja JavaScript: esperamos a que aparezca el primer <article>
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.TAG_NAME, "article"))
)
time.sleep(1)   # margen para que terminen de aparecer todos

**Paso 3 · Número Total de resultados en el mes de enero**

In [142]:
texto_pagina = navegador.find_element(By.TAG_NAME, "body").text
total_enero = int(re.search(r"(\d+) Resultados?", texto_pagina).group(1))
print("Resultados que dice la página:", total_enero)

Resultados que dice la página: 12


**Paso 4. De cada resultado en el mes de enero**

In [143]:
articulos = navegador.find_elements(By.TAG_NAME, "article")
print("Artículos encontrados:", len(articulos))
primero = articulos[0]
print(primero.text)

Artículos encontrados: 12
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 018-2025-PCM
Publicado: 30 de enero de 2025
Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto 
Descargar archivo


Numero de la norma

In [144]:
enlace_principal = primero.find_element(By.CSS_SELECTOR, "a.link-principal")
numero = enlace_principal.text     # "Decreto Supremo N.° 0XX-2025-PCM"
print(numero)

Decreto Supremo N.° 018-2025-PCM


Enlace de la norma

In [145]:
enlace = enlace_principal.get_attribute("href")   # Selenium lo devuelve completo, con https://www.gob.pe
print(enlace)

https://www.gob.pe/institucion/pcm/normas-legales/6529837-018-2025-pcm


Fecha de la publicacion

In [146]:
# La fecha se busca por lo que dice ("Publicado:"), no por su posición
fecha = None
for span in primero.find_elements(By.TAG_NAME, "span"):
    if span.text.startswith("Publicado:"):
        fecha = span.text.replace("Publicado:", "").strip()
print(fecha)

30 de enero de 2025


Titulo de la publicacion

In [147]:
titulo = primero.find_element(By.TAG_NAME, "p").text
print(titulo)

Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto 


Resultado de cada articulo

In [148]:
def leer_articulo(articulo):
    """Convierte un resultado de la búsqueda de gob.pe en un diccionario."""
    enlace_principal = articulo.find_element(By.CSS_SELECTOR, "a.link-principal")

    fecha = None
    for span in articulo.find_elements(By.TAG_NAME, "span"):
        if span.text.startswith("Publicado:"):
            fecha = span.text.replace("Publicado:", "").strip()

    parrafos = articulo.find_elements(By.TAG_NAME, "p")
    titulo = parrafos[0].text if parrafos else None

    return {
        "numero": enlace_principal.text,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace_principal.get_attribute("href"),
    }


for articulo in articulos:
    print(leer_articulo(articulo))

{'numero': 'Decreto Supremo N.° 018-2025-PCM', 'fecha': '30 de enero de 2025', 'titulo': 'Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto ', 'enlace': 'https://www.gob.pe/institucion/pcm/normas-legales/6529837-018-2025-pcm'}
{'numero': 'Decreto Supremo N.° 016-2025-PCM', 'fecha': '29 de enero de 2025', 'titulo': 'Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco ', 'enlace': 'https://www.gob.pe/institucion/pcm/normas-legales/6529813-016-2025-pcm'}
{'numero': 'Decreto Supremo N.° 013-2025-PCM', 'fecha': '23 de enero de 2025', 'titulo': 'Decreto Supremo que modifica el Decreto Supremo N° 003-2025-PCM, Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad ', 'enlace': 'https://www.gob.pe/instit

**Paso 5. Para todos los meses, entre el 01 de enero al 31 de mayo 2025**

In [149]:
meses = [
    ("enero",   "01-01-2025", "31-01-2025"),
    ("febrero", "01-02-2025", "28-02-2025"),
    ("marzo",   "01-03-2025", "31-03-2025"),
    ("abril",   "01-04-2025", "30-04-2025"),
    ("mayo",    "01-05-2025", "31-05-2025"),
]

In [150]:
todas_las_normas = []
verificacion = []

for nombre_mes, desde, hasta in meses:
    navegador.get(URL_BASE + f"&desde={desde}&hasta={hasta}")
    WebDriverWait(navegador, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "article"))
    )
    time.sleep(1)

    texto_pagina = navegador.find_element(By.TAG_NAME, "body").text
    total = int(re.search(r"(\d+) Resultados?", texto_pagina).group(1))

    # Se vuelven a buscar en cada página: los article de la página anterior ya no existen
    articulos = navegador.find_elements(By.TAG_NAME, "article")
    for articulo in articulos:
        fila = leer_articulo(articulo)
        fila["mes"] = nombre_mes
        todas_las_normas.append(fila)

    verificacion.append({
        "mes": nombre_mes,
        "resultados_totales": total,
        "resultados_extraidos": len(articulos),
    })
    print(nombre_mes, "listo")
    time.sleep(2)   # pausa entre páginas

enero listo
febrero listo
marzo listo
abril listo
mayo listo


In [151]:
navegador.quit()

**Paso 6. Verificacion de los resultados totales**

In [152]:
tabla_verificacion = pd.DataFrame(verificacion)
tabla_verificacion["coinciden"] = (
    tabla_verificacion["resultados_totales"] == tabla_verificacion["resultados_extraidos"]
)
tabla_verificacion

,mes,resultados_totales,resultados_extraidos,coinciden
0,enero,12,12,True
1,febrero,4,4,True
2,marzo,12,12,True
3,abril,15,15,True
4,mayo,11,11,True
